In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

print(module_path)

import numpy as np #
import torch
import torch.nn as nn
from hedging.envs import HedgeCallBS
from hedging.plot_utils import plot_portfolio_vs_option_price
from hedging.logit_normal import LogitNormal
from torch.distributions import LogNormal
from torchrl.modules import TanhNormal
from torchrl.envs import GymWrapper
from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.modules import ProbabilisticActor, SafeModule
from tensordict.nn import TensorDictModule
from torchrl.objectives import ClipPPOLoss
from torchrl.modules import ProbabilisticActor, SafeModule
from torchrl.modules import (
    ValueOperator,
    ActorValueOperator,
    NormalParamExtractor,
)
from torchrl.objectives.value import GAE
from torchrl.envs.utils import ExplorationType, set_exploration_type
from tensordict.nn import NormalParamExtractor

/Users/manu13/Desktop/PHD/DeepHedging/deep_hedging_v0


In [2]:
# --- Env Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K  = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
maturity = 1.0
r = 0.03
sigma = np.array([0.15, 0.2, 0.25])
num_paths = 100
num_steps = 250
history_len = 1
input_dim = 11
hidden_size = 64
action_dim = 1
transaction_cost = True
transaction_fee_rate = 1e-3

base_env = HedgeCallBS(S0, K, maturity, r, sigma, num_paths, num_steps, history_len=history_len,
                       transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate)
env = GymWrapper(base_env)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
act_spec = env.specs["input_spec", "full_action_spec", "action"].to(device)

In [3]:
frames_per_batch = env.num_envs * num_steps
sub_batch_num = 10
sub_batch_size = frames_per_batch // sub_batch_num
frames_per_batch, sub_batch_size

(150000, 15000)

In [4]:
# Param for PPO
clip_param = 0.2
value_coef = 0.1
entropy_coef = 0.001
# Param for GAE
gamma = 0.99
lmbda = 0.95

In [5]:
env.reset()

TensorDict(
    fields={
        done: Tensor(shape=torch.Size([600, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        observation: Tensor(shape=torch.Size([600, 1, 11]), device=cpu, dtype=torch.float32, is_shared=False),
        terminated: Tensor(shape=torch.Size([600, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size([600, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([600]),
    device=None,
    is_shared=False)

In [6]:
from torchrl.modules import TanhNormal
from torchrl.data import Bounded


class FeatureExtractor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        if x.ndim >= 3 and x.shape[-2] == 1:
            x = x.squeeze(-2)   # (600, 250, 11) instead of (600, 250, 1, 11)
        return self.net(x)  


feature_extractor = TensorDictModule(
    module=FeatureExtractor(input_dim=input_dim, hidden_dim=64),
    in_keys=["observation"],
    out_keys=["feature"],
)

policy_network = TensorDictModule(
    nn.Sequential(
        nn.Linear(64, 2 * action_dim),
        NormalParamExtractor(),  
    ),
    in_keys=["feature"],
    out_keys=["loc", "scale"],
)

action_spec = Bounded(
        low=0.0,
        high=1.0,
        shape=(action_dim,),
        dtype=torch.float,
        device=device,
    )

actor = ProbabilisticActor(
    module=policy_network,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    distribution_class=TanhNormal,
    return_log_prob=True,
    spec=action_spec
)

critic = ValueOperator(
    module=nn.Sequential(
        nn.Linear(64, 8),
        nn.Tanh(),
        nn.Linear(8, 1)
    ),
    in_keys=["feature"],
    out_keys=["state_value"],
)

model = ActorValueOperator(feature_extractor, actor, critic)


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ActorValueOperator(
    module=ModuleList(
      (0): TensorDictModule(
          module=FeatureExtractor(
            (net): Sequential(
              (0): Linear(in_features=11, out_features=64, bias=True)
              (1): ReLU()
              (2): Linear(in_features=64, out_features=64, bias=True)
              (3): ReLU()
            )
          ),
          device=cpu,
          in_keys=['observation'],
          out_keys=['feature'])
      (1): ProbabilisticActor(
          module=ModuleList(
            (0): TensorDictModule(
                module=Sequential(
                  (0): Linear(in_features=64, out_features=2, bias=True)
                  (1): NormalParamExtractor(
                    (scale_mapping): biased_softplus()
                  )
                ),
                device=cpu,
                in_keys=['feature'],
                out_keys=['loc', 'scale'])
            (1): SafeProbabilisticModule(
                in_keys=['loc', 'scale'],
                out_

In [8]:
advantage_module = GAE(
    gamma=gamma,
    lmbda=lmbda,
    value_network=model.get_value_operator(),
    shifted=True # make sure use this one for RNN
)

In [9]:
loss_module = ClipPPOLoss(
    actor_network=model.get_policy_operator(),
    critic_network=model.get_value_operator(),
    clip_epsilon=clip_param,
    entropy_coef=entropy_coef,
    value_coef=value_coef,
)

/Users/manu13/Desktop/PHD/DeepHedging/.DeepHedging/lib/python3.11/site-packages/torchrl/objectives/ppo.py:511: DeprecationWarning: 'entropy_coef' is deprecated and will be removed in torchrl v0.11. Please use 'entropy_coeff' instead.
  warnings.warn(


In [10]:
optim = torch.optim.Adam(loss_module.parameters(),lr=1e-4)

In [11]:
num_epochs = 10
num_episodes = 100

In [12]:
for epoch in range(num_epochs):
    for episode in range(num_episodes):
        env.reset(seed=epoch + 1000)
        collector = SyncDataCollector(
            env,
            model.get_policy_operator(),
            frames_per_batch=frames_per_batch,
            total_frames=frames_per_batch,
            device=device,
        )
        replay_buffer = ReplayBuffer(
            storage=LazyTensorStorage(max_size=frames_per_batch),
            sampler=SamplerWithoutReplacement(),
        )
        for batch in collector:
            advantage_module(batch)
            replay_buffer.extend(batch.reshape(-1).cpu())
            for _ in range(sub_batch_num):
                subdata = replay_buffer.sample(sub_batch_size)
                optim.zero_grad()
                # Forward pass PPO loss
                loss = loss_module(subdata.to(device))
                loss_critic, loss_objective, loss_entropy = (
                    loss["loss_critic"],
                    loss["loss_objective"],
                    loss["loss_entropy"],
                )
                loss_sum = loss_critic + loss_objective + loss_entropy
                # Backward pass
                loss_sum.backward()
                torch.nn.utils.clip_grad_norm_(loss_module.parameters(), max_norm=1.0)
                for param in loss_module.parameters():
                    if param.grad is not None:
                        param.grad = torch.nan_to_num(param.grad)
                # Update the networks
                optim.step()

        if (episode + 1) % 10 == 0:
            print(
                f"""Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, Loss: {loss_sum.item()}, Loss Critic: {loss_critic.item()}, Loss Obj. {loss_objective.item()}, Loss Ent. {loss_entropy.item()}, Avg. Reward: {batch['next', 'reward'].mean().item()}"""
            )
            print(batch['action'].min(), batch['action'].max(), batch['action'].mean(), batch['action'].std())

Epoch 1/10, Episode 10/100, Loss: 460.85247802734375, Loss Critic: 230.17770385742188, Loss Obj. 230.6754150390625, Loss Ent. -0.0006462935707531869, Avg. Reward: -13.784944534301758
tensor(-0.9999) tensor(1.0000) tensor(0.0669) tensor(0.6415)
Epoch 1/10, Episode 20/100, Loss: 495.8907165527344, Loss Critic: 247.68838500976562, Loss Obj. 248.20297241210938, Loss Ent. -0.000639026693534106, Avg. Reward: -15.014871597290039
tensor(-0.9997) tensor(1.0000) tensor(0.0924) tensor(0.6387)
Epoch 1/10, Episode 30/100, Loss: 463.68792724609375, Loss Critic: 231.592529296875, Loss Obj. 232.09603881835938, Loss Ent. -0.0006319473031908274, Avg. Reward: -14.22745132446289
tensor(-0.9999) tensor(1.0000) tensor(0.1253) tensor(0.6308)
Epoch 1/10, Episode 40/100, Loss: 442.0621337890625, Loss Critic: 220.7709197998047, Loss Obj. 221.29183959960938, Loss Ent. -0.0006215827888809144, Avg. Reward: -13.428502082824707
tensor(-0.9997) tensor(1.0000) tensor(0.1734) tensor(0.6128)
Epoch 1/10, Episode 50/100, 

In [13]:
# Test

base_env = HedgeCallBS(S0, K, maturity, r, sigma, 5, num_steps, history_len=history_len)
env = GymWrapper(base_env, device=device)
env.reset(seed=0)

with set_exploration_type(ExplorationType.DETERMINISTIC):
    rollout = env.rollout(max_steps=num_steps, policy=model.get_policy_operator())

In [14]:
rewards = rollout['next', 'reward'].detach().cpu().numpy()
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(np.float32(-53.149666),
 np.float32(-0.00088675297),
 np.float32(-6.049511),
 np.float32(8.0858755))

In [15]:
plot_portfolio_vs_option_price(env._env)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed